In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

#  **XGBoost Model**

In [2]:
# Not to use 'group. Instead use 'milkperiod','zdate','zdate_month'
df = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols = ["milkperiod","zdate","zdate_month","firstmilk","firstmilkdays",
                                                                "prelendays","drylendays","milkdays",
                                                                "previous_Milk305",
                                                                "firstmilk_previous","SCS_305"])

In [ ]:
df['drylendays'] = df['drylendays'].fillna(df['drylendays'].median())
df['milkdays'] = df['milkdays'].fillna(df['milkdays'].median())

In [4]:
# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float) 

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 ~ 15/(100-15)


In [5]:
numerical_columns = ["milkperiod","zdate","zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train[numerical_columns])

# Transform train, validation, and test sets
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

### **Define functions for additional metrics**


In [7]:
def calculate_mpe(y_true, y_pred):
    # Avoid division by zero
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    # Avoid division by zero
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

In [8]:
print("\n=== XGBoost Model ===")

# Train XGBoost model with default parameters
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train[numerical_columns], y_train)


=== XGBoost Model ===


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

# Predict on test set


In [9]:
y_test_pred_xgb = xgb_model.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_xgb = r2_score(y_test, y_test_pred_xgb)
test_mae_xgb = mean_absolute_error(y_test, y_test_pred_xgb)
test_mse_xgb = mean_squared_error(y_test, y_test_pred_xgb)
test_rmse_xgb = np.sqrt(test_mse_xgb)
test_mpe_xgb = calculate_mpe(y_test, y_test_pred_xgb)
test_smape_xgb = calculate_smape(y_test, y_test_pred_xgb)
test_sdr_xgb = calculate_sdr(y_test, y_test_pred_xgb)

# Print results
print("Test Set (XGBoost):")
print(f"R²    : {test_r2_xgb:.4f}")
print(f"MAE   : {test_mae_xgb:.4f}")
print(f"RMSE  : {test_rmse_xgb:.4f}")
print(f"MPE   : {test_mpe_xgb:.4f}")
print(f"sMAPE : {test_smape_xgb:.4f}")
print(f"SDR   : {test_sdr_xgb:.4f}")

Test Set (XGBoost):
R²    : 0.3668
MAE   : 6.6914
RMSE  : 8.8266
MPE   : -7.2903
sMAPE : 16.9060
SDR   : 0.6178


In [ ]:
# Predict on all sets
y_train_pred_xgb = xgb_model.predict(X_train[numerical_columns])
y_val_pred_xgb = xgb_model.predict(X_val[numerical_columns])
y_test_pred_xgb = xgb_model.predict(X_test[numerical_columns])

# Calculate metrics
train_r2_xgb = r2_score(y_train, y_train_pred_xgb)
val_r2_xgb = r2_score(y_val, y_val_pred_xgb)
test_r2_xgb = r2_score(y_test, y_test_pred_xgb)
train_mae_xgb = mean_absolute_error(y_train, y_train_pred_xgb)
val_mae_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
test_mae_xgb = mean_absolute_error(y_test, y_test_pred_xgb)
train_mse_xgb = mean_squared_error(y_train, y_train_pred_xgb)
val_mse_xgb = mean_squared_error(y_val, y_val_pred_xgb)
test_mse_xgb = mean_squared_error(y_test, y_test_pred_xgb)

# Print results
print("Train Set (XGBoost):")
print(f"R²   : {train_r2_xgb:.4f}")
print(f"MAE  : {train_mae_xgb:.4f}")
print(f"MSE  : {train_mse_xgb:.4f}")
print("\nValidation Set (XGBoost):")
print(f"R²   : {val_r2_xgb:.4f}")
print(f"MAE  : {val_mae_xgb:.4f}")
print(f"MSE  : {val_mse_xgb:.4f}")
print("\nTest Set (XGBoost):")
print(f"R²   : {test_r2_xgb:.4f}")
print(f"MAE  : {test_mae_xgb:.4f}")
print(f"MSE  : {test_mse_xgb:.4f}")

Train Set (XGBoost):
R²   : 0.4168
MAE  : 6.3925
MSE  : 70.7322

Validation Set (XGBoost):
R²   : 0.3565
MAE  : 6.6982
MSE  : 78.3261

Test Set (XGBoost):
R²   : 0.3668
MAE  : 6.6914
MSE  : 77.9084


In [ ]:
# Overfitting Check
r2_gap_xgb = train_r2_xgb - val_r2_xgb
print("\nOverfitting Analysis (XGBoost):")
print("=" * 50)
if r2_gap_xgb > 0.05:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_xgb:.4f}) is larger than threshold (0.05).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_xgb:.4f}).")


Overfitting Analysis (XGBoost):


In [ ]:
# Load existing results from CSV
comparison_df = pd.read_csv('model_comparison_table.csv', index_col=0)

# Add new XGBoost results
new_results = {
    'XGBoost': {
        'Train_R2': train_r2_xgb, 'Train_MAE': train_mae_xgb, 'Train_MSE': train_mse_xgb,
        'Val_R2': val_r2_xgb, 'Val_MAE': val_mae_xgb, 'Val_MSE': val_mse_xgb,
        'Test_R2': test_r2_xgb, 'Test_MAE': test_mae_xgb, 'Test_MSE': test_mse_xgb
    }
}

# Convert new results to DataFrame and append to existing DataFrame
new_df = pd.DataFrame.from_dict(new_results, orient='index')
comparison_df = pd.concat([comparison_df, new_df])

# Save updated results back to CSV
comparison_df.to_csv('model_comparison_table.csv', index=True)

In [ ]:
# Display updated table
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)


=== Updated Model Comparison Table ===
                      Train_R2  Train_MAE  Train_MSE    Val_R2   Val_MAE  \
RF_Initial            0.908400   2.525600  11.109300  0.340400  6.817900   
RF_Tuned              0.379600   6.612000  75.245200  0.345700  6.772200   
RF_CV_Final                NaN        NaN        NaN       NaN       NaN   
RF_Initial_with_grp   0.907400   2.541200  11.225700  0.334100  6.853500   
RF_with_grp_Tuned     0.378200   6.620800  75.415400  0.344700  6.779600   
RF_with_grp_CV_Final       NaN        NaN        NaN       NaN       NaN   
SVR_linear            0.307500   6.920500  83.988000  0.300100  6.946100   
SVR_poly              0.267900   7.151600  88.790700  0.247800  7.213500   
SVR_rbf               0.369600   6.559400  76.449100  0.355900  6.643400   
XGBoost               0.416776   6.392497  70.732196  0.356527  6.698222   

                        Val_MSE   Test_R2  Test_MAE   Test_MSE  
RF_Initial            80.284800  0.347700  6.828500  80.25

#  **XGBoost Model with Hyperparameter Tuning**

In [10]:
print("\n=== XGBoost Model with Hyperparameter Tuning ===")

# Define parameter grid for XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

# Initialize XGBoost model
xgb_model = XGBRegressor(random_state=42)

# Perform Grid Search
grid_search_xgb = GridSearchCV(estimator=xgb_model,
                               param_grid=param_grid_xgb,
                               cv=5,
                               scoring='neg_mean_squared_error',
                               n_jobs=-1,
                               verbose=2)
grid_search_xgb.fit(X_train[numerical_columns], y_train)


=== XGBoost Model with Hyperparameter Tuning ===
Fitting 5 folds for each of 729 candidates, totalling 3645 fits


KeyboardInterrupt: 

In [ ]:
# Best parameters and score
best_params_xgb = grid_search_xgb.best_params_
best_val_mse_xgb = -grid_search_xgb.best_score_
print("\nBest Hyperparameters (XGBoost):")
print(f"Best Parameters: {best_params_xgb}")
print(f"Best Validation MSE: {best_val_mse_xgb:.4f}\n")

# Best Hyperparameters (XGBoost):
# Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 0.8}
# Best Validation MSE: 76.2299


Best Hyperparameters (XGBoost):
Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 0.8}
Best Validation MSE: 76.2299



In [11]:
# Use best hyperparameters from previous Grid Search
best_params_xgb = {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 0.8}

# Train final model with best hyperparameters
final_xgb_model = XGBRegressor(**best_params_xgb, random_state=42)
final_xgb_model.fit(X_train[numerical_columns], y_train)

# Predict on test set
y_test_pred_xgb_tuned = final_xgb_model.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_xgb_tuned = r2_score(y_test, y_test_pred_xgb_tuned)
test_mae_xgb_tuned = mean_absolute_error(y_test, y_test_pred_xgb_tuned)
test_mse_xgb_tuned = mean_squared_error(y_test, y_test_pred_xgb_tuned)
test_rmse_xgb_tuned = np.sqrt(test_mse_xgb_tuned)
test_mpe_xgb_tuned = calculate_mpe(y_test, y_test_pred_xgb_tuned)
test_smape_xgb_tuned = calculate_smape(y_test, y_test_pred_xgb_tuned)
test_sdr_xgb_tuned = calculate_sdr(y_test, y_test_pred_xgb_tuned)

# Print results
print("Test Set (XGBoost Tuned):")
print(f"R²    : {test_r2_xgb_tuned:.4f}")
print(f"MAE   : {test_mae_xgb_tuned:.4f}")
print(f"RMSE  : {test_rmse_xgb_tuned:.4f}")
print(f"MPE   : {test_mpe_xgb_tuned:.4f}")
print(f"sMAPE : {test_smape_xgb_tuned:.4f}")
print(f"SDR   : {test_sdr_xgb_tuned:.4f}")

Test Set (XGBoost Tuned):
R²    : 0.3730
MAE   : 6.6601
RMSE  : 8.7831
MPE   : -7.3188
sMAPE : 16.8218
SDR   : 0.6064


In [ ]:
# Train final model with best hyperparameters
final_xgb_model = XGBRegressor(**best_params_xgb, random_state=42)
final_xgb_model.fit(X_train[numerical_columns], y_train)

# Predict on all sets
y_train_pred_xgb_tuned = final_xgb_model.predict(X_train[numerical_columns])
y_val_pred_xgb_tuned = final_xgb_model.predict(X_val[numerical_columns])
y_test_pred_xgb_tuned = final_xgb_model.predict(X_test[numerical_columns])

# Calculate metrics
train_r2_xgb_tuned = r2_score(y_train, y_train_pred_xgb_tuned)
val_r2_xgb_tuned = r2_score(y_val, y_val_pred_xgb_tuned)
test_r2_xgb_tuned = r2_score(y_test, y_test_pred_xgb_tuned)
train_mae_xgb_tuned = mean_absolute_error(y_train, y_train_pred_xgb_tuned)
val_mae_xgb_tuned = mean_absolute_error(y_val, y_val_pred_xgb_tuned)
test_mae_xgb_tuned = mean_absolute_error(y_test, y_test_pred_xgb_tuned)
train_mse_xgb_tuned = mean_squared_error(y_train, y_train_pred_xgb_tuned)
val_mse_xgb_tuned = mean_squared_error(y_val, y_val_pred_xgb_tuned)
test_mse_xgb_tuned = mean_squared_error(y_test, y_test_pred_xgb_tuned)

# Print results
print("Train Set (XGBoost Tuned):")
print(f"R²   : {train_r2_xgb_tuned:.4f}")
print(f"MAE  : {train_mae_xgb_tuned:.4f}")
print(f"MSE  : {train_mse_xgb_tuned:.4f}")
print("\nValidation Set (XGBoost Tuned):")
print(f"R²   : {val_r2_xgb_tuned:.4f}")
print(f"MAE  : {val_mae_xgb_tuned:.4f}")
print(f"MSE  : {val_mse_xgb_tuned:.4f}")
print("\nTest Set (XGBoost Tuned):")
print(f"R²   : {test_r2_xgb_tuned:.4f}")
print(f"MAE  : {test_mae_xgb_tuned:.4f}")
print(f"MSE  : {test_mse_xgb_tuned:.4f}")

Train Set (XGBoost Tuned):
R²   : 0.3899
MAE  : 6.5326
MSE  : 73.9957

Validation Set (XGBoost Tuned):
R²   : 0.3640
MAE  : 6.6596
MSE  : 77.4109

Test Set (XGBoost Tuned):
R²   : 0.3730
MAE  : 6.6601
MSE  : 77.1433


In [ ]:
# Overfitting Check
r2_gap_xgb_tuned = train_r2_xgb_tuned - val_r2_xgb_tuned
print("\nOverfitting Analysis (XGBoost Tuned):")
print("=" * 50)
if r2_gap_xgb_tuned > 0.05:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_xgb_tuned:.4f}) is larger than threshold (0.05).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_xgb_tuned:.4f}).")


Overfitting Analysis (XGBoost Tuned):
No significant overfitting based on Train-Val R² Gap (0.0258).


In [ ]:
# Load existing results from CSV
comparison_df = pd.read_csv('model_comparison_table.csv', index_col=0)

# Add new XGBoost tuned results
new_results_tuned = {
    'XGBoost_Tuned': {
        'Train_R2': train_r2_xgb_tuned, 'Train_MAE': train_mae_xgb_tuned, 'Train_MSE': train_mse_xgb_tuned,
        'Val_R2': val_r2_xgb_tuned, 'Val_MAE': val_mae_xgb_tuned, 'Val_MSE': val_mse_xgb_tuned,
        'Test_R2': test_r2_xgb_tuned, 'Test_MAE': test_mae_xgb_tuned, 'Test_MSE': test_mse_xgb_tuned
    }
}

# Convert new results to DataFrame and append to existing DataFrame
new_df_tuned = pd.DataFrame.from_dict(new_results_tuned, orient='index')
comparison_df = pd.concat([comparison_df, new_df_tuned])

# Save updated results back to CSV
comparison_df.to_csv('model_comparison_table.csv', index=True)

In [ ]:
# Display updated table
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)


=== Updated Model Comparison Table ===
                      Train_R2  Train_MAE  Train_MSE    Val_R2   Val_MAE  \
RF_Initial            0.908400   2.525600  11.109300  0.340400  6.817900   
RF_Tuned              0.379600   6.612000  75.245200  0.345700  6.772200   
RF_CV_Final                NaN        NaN        NaN       NaN       NaN   
RF_Initial_with_grp   0.907400   2.541200  11.225700  0.334100  6.853500   
RF_with_grp_Tuned     0.378200   6.620800  75.415400  0.344700  6.779600   
RF_with_grp_CV_Final       NaN        NaN        NaN       NaN       NaN   
SVR_linear            0.307500   6.920500  83.988000  0.300100  6.946100   
SVR_poly              0.267900   7.151600  88.790700  0.247800  7.213500   
SVR_rbf               0.369600   6.559400  76.449100  0.355900  6.643400   
XGBoost               0.416776   6.392497  70.732196  0.356527  6.698222   
XGBoost_Tuned         0.389867   6.532592  73.995676  0.364046  6.659580   

                        Val_MSE   Test_R2  Test

In [12]:
## **Print results into a new CSV file**
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing test evaluation table found.")
        return pd.DataFrame()

# Load existing CSV content from new file (if exists)
new_file_path = 'test_evaluation_metrics.csv'
existing_df = print_csv_content(new_file_path)

# Save results to a new CSV file
xgb_results = {
    'XGBoost': {
        'Test_R2': test_r2_xgb, 'Test_MAE': test_mae_xgb, 'Test_RMSE': test_rmse_xgb,
        'Test_MPE': test_mpe_xgb, 'Test_sMAPE': test_smape_xgb, 'Test_SDR': test_sdr_xgb
    }
}
xgb_df = pd.DataFrame.from_dict(xgb_results, orient='index')

xgb_tuned_results = {
    'XGBoost_Tuned': {
        'Test_R2': test_r2_xgb_tuned, 'Test_MAE': test_mae_xgb_tuned, 'Test_RMSE': test_rmse_xgb_tuned,
        'Test_MPE': test_mpe_xgb_tuned, 'Test_sMAPE': test_smape_xgb_tuned, 'Test_SDR': test_sdr_xgb_tuned
    }
}
xgb_tuned_df = pd.DataFrame.from_dict(xgb_tuned_results, orient='index')

# Combine new results
new_results_df = pd.concat([xgb_df, xgb_tuned_df])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the new file
comparison_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(comparison_df)


=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===
                       Test_R2  Test_MAE  Test_RMSE  Test_MPE  Test_sMAPE  \
XGBoost               0.366754  6.691355   8.826576 -7.290271   16.906020   
XGBoost_Tuned         0.372973  6.660095   8.783126 -7.318830   16.821751   
SVR_Linear            0.307137  6.970674   9.232719 -9.731248   17.584111   
SVR_Polynomial        0.264797  7.216206   9.510640 -9.820527   18.176112   
SVR_RBF               0.364292  6.637658   8.843716 -9.237821   16.758594   
SVR_RBF_Robust        0.360090  6.643834   8.861028 -9.240320   16.775699   
RF_Initial            0.347709  6.828523   8.958325 -6.936344   17.222969   
RF_Tuned              0.357528  6.759105   8.890640 -7.315823   17.055061   
RF_CV_Final           0.352554  6.784989   8.924994 -7.610628   17.092712   
RF_Initial_with_grp   0.340054  6.878539   9.010737 -6.964281   17.342442   
RF_with_grp           0.351373  6.792523   8.933128 -7.608877   17.112260  